In [ ]:
import os
import gc
import torch
from PIL import Image
from transformers import (
    AutoTokenizer,
    AutoProcessor,
    AutoModelForImageTextToText
)

# ================== Clean memory ==================
gc.collect()
torch.cuda.empty_cache()

# ================== Model config ==================
MODEL_PATH = "nanonets/Nanonets-OCR2-3B"
DTYPE = torch.float16

# ================== Load model ==================
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    device_map="auto"
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
processor = AutoProcessor.from_pretrained(MODEL_PATH)

# ================== OCR single image ==================
def ocr_image(image_path: str, max_new_tokens: int = 2048) -> str:
    prompt = (
        "Extract the text from the above document as if you were reading it naturally. "
        "Return the tables in html format. "
        "Return the equations in LaTeX representation. "
        "If there is an image in the document and image caption is not present, "
        "add a small description of the image inside the <img></img> tag; otherwise, "
        "add the image caption inside <img></img>. "
        "Watermarks should be wrapped in brackets. "
        "Ex: <watermark>OFFICIAL COPY</watermark>. "
        "Page numbers should be wrapped in brackets. "
        "Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. "
        "Prefer using ☐ and ☑ for check boxes."
    )

    image = Image.open(image_path).convert("RGB")

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{image_path}"},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    gen_ids = output_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        gen_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return output_text[0]

# ================== Process folder ==================
def process_folder(
    folder_A: str,
    folder_B: str,
    max_new_tokens: int = 2048
):
    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

    for root, _, files in os.walk(folder_A):
        rel_path = os.path.relpath(root, folder_A)
        target_dir = os.path.join(folder_B, rel_path)
        os.makedirs(target_dir, exist_ok=True)

        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext not in IMAGE_EXTS:
                continue

            img_path = os.path.join(root, file)
            md_name = os.path.splitext(file)[0] + ".md"
            md_path = os.path.join(target_dir, md_name)

            if os.path.exists(md_path):
                continue

            try:
                print(f"OCR: {img_path}")
                text = ocr_image(
                    img_path,
                    max_new_tokens=max_new_tokens
                )

                with open(md_path, "w", encoding="utf-8") as f:
                    f.write(text)

            except Exception as e:
                print(f"Lỗi {img_path}: {e}")

            torch.cuda.empty_cache()

# ================== Run ==================
if __name__ == "__main__":
    folder_A = "/kaggle/input/quydinh-quyche-haui/Data_Kaggle"
    folder_B = "/kaggle/working/Txt"

    process_folder(
        folder_A=folder_A,
        folder_B=folder_B,
        max_new_tokens=2048
    )


2026-02-01 06:24:13.971345: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769927054.160964      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769927054.214531      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769927054.666829      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769927054.666872      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769927054.666874      55 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_3.png


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_9.png
OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_8.png
OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_1.png
OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_2.png
OCR: /kaggle/input/quydinh-quyche-haui/Data_Kaggle/KhenThuongKyLuat/page_7.png


In [2]:
cd /kaggle/working


/kaggle/working


In [3]:
!zip -r Txt.zip Txt


  adding: Txt/ (stored 0%)
  adding: Txt/CongNhanMonHocTamThoi/ (stored 0%)
  adding: Txt/CongNhanMonHocTamThoi/page_8.md (deflated 74%)
  adding: Txt/CongNhanMonHocTamThoi/page_13.md (deflated 78%)
  adding: Txt/CongNhanMonHocTamThoi/page_14.md (deflated 72%)
  adding: Txt/CongNhanMonHocTamThoi/page_7.md (deflated 36%)
  adding: Txt/CongNhanMonHocTamThoi/page_1.md (deflated 65%)
  adding: Txt/CongNhanMonHocTamThoi/page_10.md (deflated 71%)
  adding: Txt/CongNhanMonHocTamThoi/page_19.md (deflated 56%)
  adding: Txt/CongNhanMonHocTamThoi/page_18.md (deflated 54%)
  adding: Txt/CongNhanMonHocTamThoi/page_17.md (deflated 53%)
  adding: Txt/CongNhanMonHocTamThoi/page_6.md (deflated 69%)
  adding: Txt/CongNhanMonHocTamThoi/page_4.md (deflated 66%)
  adding: Txt/CongNhanMonHocTamThoi/page_2.md (deflated 32%)
  adding: Txt/CongNhanMonHocTamThoi/page_12.md (deflated 75%)
  adding: Txt/CongNhanMonHocTamThoi/page_9.md (deflated 72%)
  adding: Txt/CongNhanMonHocTamThoi/page_20.md (deflated 58%)
 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
